# 🇹🇭 Thai NLP Toolkit — Training on Colab (Simplified Version)

Notebook นี้ออกแบบมาให้ใช้งานง่ายที่สุดโดย **ไม่ต้องสร้างไฟล์ Zip หรือใช้ Google Drive**:
1. Clone โปรเจคจาก GitHub
2. ติดตั้ง library
3. ดาวน์โหลด datasets ล่าสุดจาก Hugging Face (อัตโนมัติ)
4. อัปโหลดเฉพาะโมเดล Tokenizer (2 ไฟล์สั้นๆ)
5. เริ่มต้นเทรนด้วย GPU

> ⚠️ **ก่อนเริ่ม**: ไปที่เมนูด้านบน **Runtime (รันไทม์) → Change runtime type (เปลี่ยนประเภทการรันไทม์) → GPU (T4)**

## 1. ตรวจสอบ GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Clone โปรเจคจาก GitHub

In [ ]:
!git clone https://github.com/puttibenz/thai-nlp-toolkit.git
%cd thai-nlp-toolkit

## 3. ติดตั้ง Dependencies

In [ ]:
!pip install -q sentencepiece>=0.1.99 pythainlp>=4.0.0 datasets>=2.14.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0

## 4. ดาวน์โหลด Raw Datasets (อัตโนมัติ)

รันเซลล์นี้เพื่อดาวน์โหลดข้อมูลจาก Hugging Face และจัดสรรลงโฟลเดอร์ `data/raw/` โดยตรง

In [ ]:
!python data/download.py --task all

## 5. อัปโหลดโมเดล Tokenizer

เนื่องจากโมเดล BPE Tokenizer ไม่สามารถก๊อปปี้ผ่าน GitHub ได้ ให้รันเซลล์ด้านล่างนี้แล้วเลือกอัปโหลดไฟล์จากเครื่องคอมพิวเตอร์ของคุณ:
1. **`thai_bpe.model`** (ขนาด ~250 KB)
2. **`thai_bpe.vocab`** (ขนาด ~5 KB)

*(ทั้งสองไฟล์อยู่ในโฟลเดอร์ `tokenizer` บนเครื่องของคุณ)*

In [ ]:
import os
from google.colab import files

print("กรุณาเลือกไฟล์ 'thai_bpe.model' และ 'thai_bpe.vocab' เพื่ออัปโหลด:")
uploaded = files.upload()

os.makedirs("tokenizer", exist_ok=True)
for filename in uploaded.keys():
    dest = os.path.join("tokenizer", filename)
    if os.path.exists(dest):
        os.remove(dest)
    os.rename(filename, dest)
    print(f"✅ ย้าย {filename} เข้าโฟลเดอร์ tokenizer/ สำเร็จ")

## 6. ตรวจสอบความพร้อมของไฟล์

In [ ]:
import os

required_files = [
    "tokenizer/thai_bpe.model",
    "tokenizer/thai_bpe.vocab",
    "tokenizer/tokenizer_config.json",
    "data/raw/ner_train.jsonl",
    "data/raw/sent_train.tsv",
    "data/raw/qa_train.json",
]

for f in required_files:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status}  {f}")

## 7. เริ่มต้น Training 🚀

In [ ]:
!python -m scripts.train --device cuda

## 8. บันทึกผลลัพธ์กลับ Google Drive

หลังจากรันเสร็จแล้ว สามารถบันทึกโฟลเดอร์ checkpoints กลับเข้า Google Drive ได้ที่นี่:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r outputs/ /content/drive/MyDrive/thai_nlp_outputs/
print("✅ สำรองข้อมูลโมเดลไปยัง Google Drive เรียบร้อยแล้ว")